# RAG walkthrough: start here

*Rulebook notebooks: **Start here** · Deep dive 1: <a href="https://ecoop.github.io/rulebook/embeddings_and_eval.html" target="_parent">Embeddings & evaluation</a>*

> **Two ways to use this notebook**
> - **Read it, with results:** the <a href="https://ecoop.github.io/rulebook/walkthrough.html" target="_parent">results page</a> shows it after a fresh run,
>   Claude's answers included.
> - **Run it on your machine:** see **Before you start** below. Section 3 needs an Anthropic API key,
>   because it asks Claude to write answers (three questions, about a cent each).
>
> GitHub's own notebook view shows the code only: no results, and nothing runs there.
>
> **No key?** <a href="https://ecoop.github.io/rulebook/embeddings_and_eval.html" target="_parent">Deep dive 1</a> needs none, and
> <a href="https://colab.research.google.com/github/ecoop/rulebook/blob/main/notebooks/embeddings_and_eval.ipynb" target="_parent">runs free in Colab</a>.

A quick tour of Retrieval-Augmented Generation: one look at each stage as a question goes
through the pipeline, **chunk → embed and retrieve → generate**, on two small sample rulebooks
(`checkers` and `cornhole`). Each stage points to a **deep dive** that slows down and explains it
properly:

- **Deep dive 1: <a href="https://ecoop.github.io/rulebook/embeddings_and_eval.html" target="_parent">Embeddings & evaluation</a>.** What an embedding is,
  and how to tell whether retrieval is any good (golds, hit@k, MRR). Key-free.
- *Planned:* chunking · reranking (<a href="https://github.com/ecoop/rulebook/issues/258" target="_parent">#258</a>) · generation and citations.

**Before you start:** `uv sync`, copy `.env.example` to `.env` and set `ANTHROPIC_API_KEY`, then
run `uv run jupyter lab` from the repo root. The first cell builds the sample index if it's missing.

In [ ]:
import os, pathlib, sys
# Work from the repo root (where pyproject.toml lives) so .env, rules/ and data/ resolve.
root = pathlib.Path.cwd()
while not (root / 'pyproject.toml').exists() and root != root.parent:
    root = root.parent
os.chdir(root)
sys.path.insert(0, str(root / 'notebooks'))  # nb_helpers.py, shared by the notebooks
from dotenv import load_dotenv
load_dotenv(root / '.env')

from rulebook.app_state import initialize
from rulebook.config import settings
from nb_helpers import ensure_index, show_table

initialize(settings)  # the cost tracking that embedding and generation calls report to
ensure_index('checkers', 'cornhole')

## 1. Chunk

RAG retrieves *pieces* of documents, not whole files, and how those pieces are cut is one of the
biggest levers on the whole system. Here the chunker splits each rulebook on its own rule numbers,
so each chunk is one rule you can cite.

In [ ]:
from rulebook.ingest import extract_pages
from rulebook.chunking import chunk_pages

doc = pathlib.Path('rules/checkers/checkers.md')
chunks = chunk_pages(extract_pages(doc), source=doc.name, domain='checkers')

print(f'{len(chunks)} chunks from {doc.name}; the first five:')
show_table(['rule', 'text'], [(c.rule_id, c.text[:80] + '…') for c in chunks[:5]])

The `rule` id on each chunk is what an answer cites, so a reader can check a claim against the
actual rule. Text before the first numbered rule becomes a `preamble` chunk; Deep dive 1 (§4)
shows how that one can get in the way.

## 2. Embed and retrieve

Each chunk was turned into an **embedding**: a vector placed so that similar meanings land near
each other. To answer a question, embed it the same way and take the nearest chunks.
**Distance** is 2 − 2 × the cosine similarity of the two vectors: 0 means identical,
2 unrelated, 4 opposite. Lower is nearer.

In [ ]:
from rulebook.retrieve import retrieve

hits = retrieve('can a king move backwards?', domain='checkers', k=3)
show_table(['rank', 'rule', 'distance', 'text'],
           [(i, h.rule_id, f'{h.distance:.2f}', h.text[:80] + '…') for i, h in enumerate(hits, 1)])

Rule 4.2, *How a king moves*, comes first, though the question barely shares its words.

**Deep dive 1** (<a href="https://ecoop.github.io/rulebook/embeddings_and_eval.html" target="_parent">Embeddings & evaluation</a>) looks inside an
embedding, shows a question where the nearest chunk isn't the answer, and measures retrieval
with a gold set instead of by eye.

## 3. Generate

`ask` runs the whole pipeline: retrieve, then give Claude *only* those chunks, with instructions
to cite every claim as `[domain rule]` and to refuse anything the chunks don't cover.
`unverified_citations` flags any rule the answer cites that wasn't among them.

In [ ]:
assert os.getenv('ANTHROPIC_API_KEY'), 'This section needs ANTHROPIC_API_KEY in .env'
from rulebook.pipeline import ask

res = ask('How does a piece become a king, and what can it then do?', domain='checkers')
print(res.answer)
print('\nretrieved:', [f'{c.domain} {c.rule_id}' for c in res.chunks])
print('unverified citations:', res.unverified_citations)

**The guardrail.** Ask about something the rules don't contain. Instead of answering from what
the model already knows, it says the rules don't cover it.

In [ ]:
res = ask('What is the offside rule in soccer?', domain='checkers')
print(res.answer)

**Two rulebooks at once.** `ask` can retrieve from several domains and compare them.

In [ ]:
res = ask('How do you win each game?', domains=['checkers', 'cornhole'])
print(res.answer)
print('\nretrieved from:', sorted({c.domain for c in res.chunks}))

## Where to go next

- **Deep dive 1: <a href="https://ecoop.github.io/rulebook/embeddings_and_eval.html" target="_parent">Embeddings & evaluation</a>** (<a href="https://colab.research.google.com/github/ecoop/rulebook/blob/main/notebooks/embeddings_and_eval.ipynb" target="_parent">run it in Colab</a>, no key needed).
- **Turn a knob.** `ask` hands Claude the 5 nearest chunks; pass `k=1` or `k=8` in §3 and watch
  the answer change. Or edit a rule in `rules/checkers/checkers.md`, rebuild, and ask again.
- **Add a real ruleset.** `uv run python -m rulebook.add <slug> <your.pdf>` ingests and indexes a
  rule document; `rules/SOURCES.md` lists recognizable ones.
- **Read the core modules** in order: `ingest` → `chunking` → `embeddings` → `store` → `retrieve` →
  `generate`. The rest of `src/rulebook/` (auth, roles, cost tracking) is app scaffolding.